# Model lead: steps 1–2 on a free Colab T4

Runtime → Change runtime type → **T4 GPU** first. Run the cells top to bottom.
Images live outside git (shared Drive), so cell 3 points `data/matched/` at the Drive folder.
Embeddings are written to Drive too, so a disconnected session doesn't lose them.

In [ ]:
# 1. GPU?
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: switch the runtime to T4")

In [ ]:
# 2. Get the code (the branch with the Model work)
REPO = "https://github.com/huynietran/ai-image-detector.git"
BRANCH = "model/interface-step1-step2"   # "main" once the PR is merged
!git clone -q -b {BRANCH} {REPO} /content/ai-image-detector || (cd /content/ai-image-detector && git pull -q)
%cd /content/ai-image-detector
!pip -q install transformers peft wandb pyyaml scikit-learn pillow

In [ ]:
# 3. Point data/matched/ and model/embeddings/ at the shared Drive folder
from google.colab import drive; drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ai-image-detector"   # <- the team's shared folder; same layout as data/
import os, shutil
os.makedirs(f"{DRIVE}/model/embeddings", exist_ok=True)
for local, remote in [("data/matched", f"{DRIVE}/data/matched"), ("model/embeddings", f"{DRIVE}/model/embeddings")]:
    if os.path.islink(local): continue
    shutil.rmtree(local, ignore_errors=True); os.symlink(remote, local)
!git pull -q   # latest manifest (Data's v0, Evaluation's freeze)
import pandas as pd
m = pd.read_csv("data/manifests/manifest.csv")
print(len(m), "manifest rows,", len(os.listdir("data/matched")), "files in data/matched")
print(m.groupby(["split", "is_ai", "generator"], dropna=False).size())

In [ ]:
# 4. Smoke test: --tiny proves the plumbing in ~1 min; without it, the first run on real weights
!python model/steps/smoke_test_model.py --tiny
# !python model/steps/smoke_test_model.py

In [ ]:
# 5. Step 1: embeddings (~15-30 min for 10k images on a T4). Re-running only embeds new rows.
!python model/steps/step1_extract_embeddings.py --config model/configs/step1.yaml
import numpy as np
print(np.load("model/embeddings/dinov2_vitb14.npz")["emb"].shape, "-> should be (train + held-out rows, 5, 768)")

**Stop here until Evaluation's freeze commit is in the manifest** (`split == held_out_gen` rows exist).
Step 2 refuses to run if a held-out generator is still in train.

In [ ]:
# 6. Step 2: the baseline (seconds)
!git pull -q
!python model/steps/step2_baseline_logreg.py --config model/configs/step2.yaml
!cat evaluation/results/step2.json

In [ ]:
# 7. Share the checkpoint (W&B artifact + SHA-256 for Evaluation's cross-check)
import wandb; wandb.login()
!python model/steps/share_checkpoint.py --step 2
shutil.copy("model/checkpoints/step2.pt", f"{DRIVE}/model/step2.pt")   # Drive copy too

In [ ]:
# 8. Step 3, only after the step-2 number is in the results table
# !python model/steps/step3_add_augmentation.py --config model/configs/step3.yaml